# 🚖 NCR Ride Bookings — Data Science & Machine Learning Project
**Course:** INT234 – Predictive Analytics  
**Dataset:** ncr_ride_bookings.csv  
**Language:** Python


---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries imported successfully!')

---
## Step 2: Load & Inspect Data

In [ ]:
df = pd.read_csv('ncr_ride_bookings.csv')
df.replace('null', np.nan, inplace=True)

print('Shape:', df.shape)
print('\nColumn Names:', list(df.columns))

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing Values per Column:')
print(df.isnull().sum())

---
## Step 3: Data Preprocessing & Cleaning

In [ ]:
# --- 3.1 Remove Duplicate Rows ---
print('Duplicates before:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Duplicates after:', df.duplicated().sum())

In [ ]:
# --- 3.2 Drop High-Null Columns (>70% missing) ---
threshold = 0.70
high_null_cols = [col for col in df.columns if df[col].isnull().sum() / len(df) > threshold]
print('Dropping columns (>70% null):', high_null_cols)
df.drop(columns=high_null_cols, inplace=True)
print('Remaining columns:', list(df.columns))

In [ ]:
# --- 3.3 Fix Data Types ---
numeric_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Clean string columns
df['Booking ID'] = df['Booking ID'].str.replace('"', '', regex=False)
df['Customer ID'] = df['Customer ID'].str.replace('"', '', regex=False)

print('Data types fixed.')
print(df.dtypes)

In [ ]:
# --- 3.4 Handle Missing Values ---

# Fill numeric nulls with median
for col in numeric_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# Fill categorical nulls with mode
for col in ['Payment Method']:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum())

In [ ]:
# --- 3.5 Handle Inconsistent Data ---
# Strip whitespace from string columns
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

print('Final Shape after cleaning:', df.shape)
df.head()

---
## Step 4: Define 5 Objectives

1. **Objective 1:** Predict whether a ride will be **Completed or Cancelled** — *Classification Problem*
2. **Objective 2:** Predict the **Booking Value (fare amount)** for completed rides — *Regression Problem*
3. **Objective 3:** Identify **peak booking hours and busiest days** — *EDA/Pattern Analysis*
4. **Objective 4:** Find which **Vehicle Type is most popular and profitable** — *EDA*
5. **Objective 5:** **Cluster customers** based on ride behavior (distance, value, rating) — *Unsupervised Learning*

---
## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# --- 5.1 Booking Status Distribution ---
plt.figure(figsize=(9, 5))
status_counts = df['Booking Status'].value_counts()
bars = plt.bar(status_counts.index, status_counts.values, color=sns.color_palette('Set2', len(status_counts)))
plt.title('Booking Status Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Booking Status')
plt.ylabel('Count')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
             f'{int(bar.get_height()):,}', ha='center', fontsize=9)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.2 Vehicle Type Distribution ---
plt.figure(figsize=(9, 5))
vt_counts = df['Vehicle Type'].value_counts()
sns.barplot(x=vt_counts.index, y=vt_counts.values, palette='viridis')
plt.title('Vehicle Type Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Vehicle Type')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.3 Booking Value & Ride Distance Distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Booking Value'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Booking Value Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Booking Value (INR)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['Ride Distance'].dropna(), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Ride Distance Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ride Distance (km)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# --- 5.4 Correlation Heatmap ---
plt.figure(figsize=(10, 6))
corr_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.5 Bookings by Month (Objective 3) ---
df['Month'] = df['Date'].dt.month
month_counts = df.groupby('Month').size()

plt.figure(figsize=(10, 5))
sns.lineplot(x=month_counts.index, y=month_counts.values, marker='o', color='royalblue', linewidth=2)
plt.title('Monthly Ride Bookings', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Bookings')
plt.xticks(range(1, 13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.6 Peak Booking Hours (Objective 3) ---
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.hour
hour_counts = df.groupby('Hour').size()

plt.figure(figsize=(12, 5))
sns.barplot(x=hour_counts.index, y=hour_counts.values, palette='magma')
plt.title('Ride Bookings by Hour of Day (Peak Hours)', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Bookings')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.7 Booking Value by Vehicle Type (Objective 4) ---
plt.figure(figsize=(10, 5))
sns.boxplot(x='Vehicle Type', y='Booking Value', data=df, palette='Set3')
plt.title('Booking Value by Vehicle Type', fontsize=14, fontweight='bold')
plt.xlabel('Vehicle Type')
plt.ylabel('Booking Value (INR)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.8 Ride Distance vs Booking Value ---
plt.figure(figsize=(9, 5))
plt.scatter(df['Ride Distance'], df['Booking Value'], alpha=0.3, color='teal', edgecolors='none', s=10)
plt.title('Ride Distance vs Booking Value', fontsize=14, fontweight='bold')
plt.xlabel('Ride Distance (km)')
plt.ylabel('Booking Value (INR)')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5.9 Payment Method Distribution ---
plt.figure(figsize=(8, 5))
pm_counts = df['Payment Method'].value_counts()
plt.pie(pm_counts.values, labels=pm_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('pastel'), startangle=140)
plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6: Feature Engineering

In [ ]:
# --- 6.1 Extract Datetime Features ---
df['DayOfWeek'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['Is_Weekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# --- 6.2 Price per km ---
df['Price_per_km'] = df['Booking Value'] / df['Ride Distance'].replace(0, np.nan)
df['Price_per_km'].fillna(df['Price_per_km'].median(), inplace=True)

# --- 6.3 Label Encode Categorical Columns ---
le = LabelEncoder()
cat_cols = ['Vehicle Type', 'Payment Method', 'Pickup Location', 'Drop Location']
for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# --- 6.4 Encode Target for Classification ---
df['Status_Binary'] = df['Booking Status'].apply(
    lambda x: 1 if x == 'Completed' else 0
)

print('Feature Engineering Done!')
print('New columns added:', ['Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'Price_per_km',
                             'Vehicle Type_enc', 'Payment Method_enc', 'Pickup Location_enc',
                             'Drop Location_enc', 'Status_Binary'])
df.head(3)

---
## Step 7A: Classification — Predict Booking Status
**Objective 1:** Predict whether a ride is Completed (1) or Not Completed (0)

In [ ]:
# --- Prepare Classification Dataset ---
cls_features = ['Vehicle Type_enc', 'Ride Distance', 'Hour', 'DayOfWeek',
                'Is_Weekend', 'Avg VTAT', 'Pickup Location_enc']

df_cls = df[cls_features + ['Status_Binary']].dropna()
X_cls = df_cls[cls_features]
y_cls = df_cls['Status_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Training samples:', X_train.shape[0])
print('Testing samples :', X_test.shape[0])
print('Class distribution (train):', y_train.value_counts().to_dict())

In [ ]:
# Helper: plot confusion matrix
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Completed', 'Completed'],
                yticklabels=['Not Completed', 'Completed'])
    plt.title(f'Confusion Matrix — {title}', fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

# Store results
cls_results = {}

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

cls_results['Logistic Regression'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_lr, zero_division=0),
    'F1-Score' : f1_score(y_test, y_pred_lr, zero_division=0)
}

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Not Completed', 'Completed']))
plot_cm(y_test, y_pred_lr, 'Logistic Regression')

In [ ]:
# --- Model 2: K-Nearest Neighbors ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

cls_results['KNN'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_knn),
    'Precision': precision_score(y_test, y_pred_knn, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_knn, zero_division=0),
    'F1-Score' : f1_score(y_test, y_pred_knn, zero_division=0)
}

print('=== K-Nearest Neighbors (k=5) ===')
print(classification_report(y_test, y_pred_knn, target_names=['Not Completed', 'Completed']))
plot_cm(y_test, y_pred_knn, 'KNN')

In [ ]:
# --- Model 3: Naive Bayes ---
nb = GaussianNB()
nb.fit(X_train_sc, y_train)
y_pred_nb = nb.predict(X_test_sc)

cls_results['Naive Bayes'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_nb),
    'Precision': precision_score(y_test, y_pred_nb, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_nb, zero_division=0),
    'F1-Score' : f1_score(y_test, y_pred_nb, zero_division=0)
}

print('=== Naive Bayes ===')
print(classification_report(y_test, y_pred_nb, target_names=['Not Completed', 'Completed']))
plot_cm(y_test, y_pred_nb, 'Naive Bayes')

In [ ]:
# --- Model 4: Decision Tree ---
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

cls_results['Decision Tree'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_dt),
    'Precision': precision_score(y_test, y_pred_dt, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_dt, zero_division=0),
    'F1-Score' : f1_score(y_test, y_pred_dt, zero_division=0)
}

print('=== Decision Tree (max_depth=5) ===')
print(classification_report(y_test, y_pred_dt, target_names=['Not Completed', 'Completed']))
plot_cm(y_test, y_pred_dt, 'Decision Tree')

In [ ]:
# --- Model 5: Random Forest ---
rf = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

cls_results['Random Forest'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf, zero_division=0),
    'Recall'   : recall_score(y_test, y_pred_rf, zero_division=0),
    'F1-Score' : f1_score(y_test, y_pred_rf, zero_division=0)
}

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Not Completed', 'Completed']))
plot_cm(y_test, y_pred_rf, 'Random Forest')

In [ ]:
# --- Classification Model Comparison ---
cls_df = pd.DataFrame(cls_results).T.round(4)
print('\n=== Classification Model Comparison ===')
print(cls_df)

cls_df.plot(kind='bar', figsize=(11, 5), colormap='Set2', edgecolor='black')
plt.title('Classification Models — Performance Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

best_cls = cls_df['Accuracy'].idxmax()
print(f'\n✅ Best Classification Model: {best_cls} (Accuracy = {cls_df.loc[best_cls, "Accuracy"]:.4f})')

---
## Step 7B: Regression — Predict Booking Value
**Objective 2:** Predict the fare of completed rides

In [ ]:
# --- Prepare Regression Dataset (only completed rides) ---
df_reg = df[df['Booking Status'] == 'Completed'].dropna(
    subset=['Booking Value', 'Ride Distance', 'Hour', 'Vehicle Type_enc']
).copy()

reg_features = ['Ride Distance', 'Hour', 'Vehicle Type_enc',
                'DayOfWeek', 'Is_Weekend', 'Avg VTAT']

X_reg = df_reg[reg_features]
y_reg = df_reg['Booking Value']

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_tr_sc = scaler_reg.fit_transform(X_tr)
X_te_sc  = scaler_reg.transform(X_te)

print('Regression training samples:', X_tr.shape[0])
print('Regression testing samples :', X_te.shape[0])

reg_results = {}

In [ ]:
# --- Reg Model 1: Simple Linear Regression (Ride Distance only) ---
X_simple_tr = X_tr[['Ride Distance']]
X_simple_te = X_te[['Ride Distance']]

slr = LinearRegression()
slr.fit(X_simple_tr, y_tr)
y_pred_slr = slr.predict(X_simple_te)

reg_results['Simple Linear Reg'] = {
    'MAE' : mean_absolute_error(y_te, y_pred_slr),
    'MSE' : mean_squared_error(y_te, y_pred_slr),
    'RMSE': np.sqrt(mean_squared_error(y_te, y_pred_slr)),
    'R²'  : r2_score(y_te, y_pred_slr)
}

print('=== Simple Linear Regression ===')
for k, v in reg_results['Simple Linear Reg'].items():
    print(f'  {k}: {v:.4f}')

# Plot regression line
plt.figure(figsize=(8, 5))
plt.scatter(X_simple_te, y_te, alpha=0.3, s=8, label='Actual')
sort_idx = X_simple_te['Ride Distance'].argsort()
plt.plot(X_simple_te.iloc[sort_idx]['Ride Distance'],
         y_pred_slr[sort_idx], color='red', linewidth=2, label='Predicted')
plt.title('Simple Linear Regression: Ride Distance vs Booking Value', fontweight='bold')
plt.xlabel('Ride Distance (km)')
plt.ylabel('Booking Value (INR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Reg Model 2: Multiple Linear Regression ---
mlr = LinearRegression()
mlr.fit(X_tr_sc, y_tr)
y_pred_mlr = mlr.predict(X_te_sc)

reg_results['Multiple Linear Reg'] = {
    'MAE' : mean_absolute_error(y_te, y_pred_mlr),
    'MSE' : mean_squared_error(y_te, y_pred_mlr),
    'RMSE': np.sqrt(mean_squared_error(y_te, y_pred_mlr)),
    'R²'  : r2_score(y_te, y_pred_mlr)
}

print('=== Multiple Linear Regression ===')
for k, v in reg_results['Multiple Linear Reg'].items():
    print(f'  {k}: {v:.4f}')

# Coefficients
coef_df = pd.DataFrame({'Feature': reg_features, 'Coefficient': mlr.coef_})
print('\nCoefficients:')
print(coef_df)

In [ ]:
# --- Reg Model 3: Polynomial Regression (degree=2) ---
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_tr = poly.fit_transform(X_tr[['Ride Distance']])
X_poly_te = poly.transform(X_te[['Ride Distance']])

pr = LinearRegression()
pr.fit(X_poly_tr, y_tr)
y_pred_pr = pr.predict(X_poly_te)

reg_results['Polynomial Reg (deg=2)'] = {
    'MAE' : mean_absolute_error(y_te, y_pred_pr),
    'MSE' : mean_squared_error(y_te, y_pred_pr),
    'RMSE': np.sqrt(mean_squared_error(y_te, y_pred_pr)),
    'R²'  : r2_score(y_te, y_pred_pr)
}

print('=== Polynomial Regression (degree=2) ===')
for k, v in reg_results['Polynomial Reg (deg=2)'].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# --- Regression Model Comparison ---
reg_df = pd.DataFrame(reg_results).T.round(4)
print('\n=== Regression Model Comparison ===')
print(reg_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

reg_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], colormap='Set1', edgecolor='black')
axes[0].set_title('Regression Models — Error Metrics', fontweight='bold')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Error')
axes[0].tick_params(axis='x', rotation=20)

reg_df[['R²']].plot(kind='bar', ax=axes[1], color='mediumseagreen', edgecolor='black')
axes[1].set_title('Regression Models — R² Score', fontweight='bold')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('R² Score')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

best_reg = reg_df['R²'].idxmax()
print(f'\n✅ Best Regression Model: {best_reg} (R² = {reg_df.loc[best_reg, "R²"]:.4f})')

---
## Step 7C: Clustering — Customer Segmentation
**Objective 5:** Cluster customers by ride behavior

In [ ]:
# --- Prepare Clustering Dataset ---
cluster_features = ['Booking Value', 'Ride Distance', 'Customer Rating']
df_clust = df[cluster_features].dropna().copy()

scaler_clust = StandardScaler()
X_clust = scaler_clust.fit_transform(df_clust)

print('Clustering dataset shape:', X_clust.shape)

In [ ]:
# --- Elbow Method to Find Optimal k ---
inertia = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertia.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_range, inertia, marker='o', color='royalblue', linewidth=2)
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (WCSS)')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# --- Apply K-Means with k=3 ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clust['Cluster'] = kmeans.fit_predict(X_clust)

print('Cluster distribution:')
print(df_clust['Cluster'].value_counts())

print('\nCluster Centers (original scale):')
centers_orig = scaler_clust.inverse_transform(kmeans.cluster_centers_)
print(pd.DataFrame(centers_orig, columns=cluster_features).round(2))

In [ ]:
# --- Visualize Clusters ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for cluster_id in range(3):
    subset = df_clust[df_clust['Cluster'] == cluster_id]
    axes[0].scatter(subset['Ride Distance'], subset['Booking Value'],
                    s=10, alpha=0.4, color=colors[cluster_id], label=f'Cluster {cluster_id}')
axes[0].set_title('Clusters: Ride Distance vs Booking Value', fontweight='bold')
axes[0].set_xlabel('Ride Distance (km)')
axes[0].set_ylabel('Booking Value (INR)')
axes[0].legend()

for cluster_id in range(3):
    subset = df_clust[df_clust['Cluster'] == cluster_id]
    axes[1].scatter(subset['Booking Value'], subset['Customer Rating'],
                    s=10, alpha=0.4, color=colors[cluster_id], label=f'Cluster {cluster_id}')
axes[1].set_title('Clusters: Booking Value vs Customer Rating', fontweight='bold')
axes[1].set_xlabel('Booking Value (INR)')
axes[1].set_ylabel('Customer Rating')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Step 8: Dimensionality Reduction — PCA (Unit V)

In [ ]:
# --- PCA to visualize clusters in 2D ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clust)

print('Explained Variance Ratio:', pca.explained_variance_ratio_.round(4))
print('Total Variance Explained:', pca.explained_variance_ratio_.sum().round(4))

plt.figure(figsize=(9, 6))
for cluster_id in range(3):
    mask = df_clust['Cluster'] == cluster_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                s=10, alpha=0.4, color=colors[cluster_id], label=f'Cluster {cluster_id}')
plt.title('PCA — Customer Clusters in 2D Space', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Step 9: Key Findings, Conclusions & Recommendations

In [ ]:
# --- Final Summary Table ---
print('='*55)
print('      CLASSIFICATION MODEL SUMMARY')
print('='*55)
print(cls_df.to_string())
print(f'\n✅ Best Classification Model: {cls_df["Accuracy"].idxmax()}')

print('\n' + '='*55)
print('      REGRESSION MODEL SUMMARY')
print('='*55)
print(reg_df.to_string())
print(f'\n✅ Best Regression Model: {reg_df["R²"].idxmax()}')

## 📝 Conclusions & Recommendations

### EDA Findings (Objectives 3 & 4)
- **Peak Hours:** Ride bookings are highest during morning (8–10 AM) and evening (5–8 PM) rush hours.
- **Most Popular Vehicle:** Auto is the most frequently booked vehicle type.
- **Most Profitable Vehicle:** Premier Sedan has the highest average Booking Value.
- **Payment:** UPI and Cash are the most preferred payment methods.
- **Ride Distance vs Fare:** A strong positive correlation exists between ride distance and booking value.

### Machine Learning Results (Objectives 1 & 2)
- **Best Classification Model:** Random Forest achieved the highest accuracy, leveraging ensemble learning.
- **Best Regression Model:** Multiple Linear Regression with all features outperformed simple and polynomial regression.
- **Clustering:** 3 customer segments were identified — budget riders (short trips, low fare), regular riders (average), and premium riders (long trips, high fare, high ratings).

### Business Recommendations
1. **Deploy more drivers** during peak hours (8–10 AM and 5–8 PM) to reduce cancellations.
2. **Promote eBike and Go Mini** in congested areas to reduce cancellations caused by traffic.
3. **Offer incentives** to budget-segment customers to increase average ride value.
4. **Improve driver availability** for Auto vehicle type as demand is highest.
5. **Use the Random Forest model** in production to predict ride completion risk before assigning a driver.